# Principal component analysis of soil gradients

Reproduces Figure 4.13: the global PCA and the scheme-specific PCAs for Perkerra and Tana. The active variables are MIR-predicted EC$_{1:2}$, ESP, pH, SOC, clay, CEC and TN; microbial indicators are projected as supplementary variables in the global PCA.

The notebook contains only the analysis retained in the final thesis. Exploratory alternatives, superseded cells, personal paths, saved outputs, and unpublished figure-formatting trials have been removed.

In [ ]:
# ============================================================
# FIGURE 4.13 — COMBINED PCA FIGURE FROM SCRATCH
# Bigger legend + bigger points + forced TN label position in panel b
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.lines import Line2D

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import pearsonr

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

file_path = PROJECT_DIR / "data" / "soil_data.xlsx"
sheet_name = "PredictionNewSamplesSalinity_Fi"

out_dir = PROJECT_DIR / "outputs" / "02_pca_soil_gradients"
tables_dir = out_dir / "tables"
figures_dir = out_dir / "figures"

for folder in [tables_dir, figures_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if not file_path.exists():
    raise FileNotFoundError(
        f"Input file not found: {file_path}\n"
        "Place the dataset in data/soil_data.xlsx or update file_path."
    )

# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------
df = pd.read_excel(file_path, sheet_name=sheet_name)
df.columns = df.columns.str.strip()

# ------------------------------------------------------------
# 3. BASIC CLEANING
# ------------------------------------------------------------
df["Zone"] = df["SITE"].astype(str).str.strip().str.upper()
df = df[df["Zone"].isin(["PERKERRA", "TANA"])].copy()

rename_dict = {
    "EC 1:2 (dS/cm)": "EC12",
    "ESP est": "ESP",
    "SOC (%)": "SOC",
    "Clay (%)": "Clay",
    "CEC cmolc/kg": "CEC",
    "TN (mg/kg)": "TN",
    "TYPES_BACTERIA": "TypesBacteria",
    "TYPES_FUNGI": "TypesFungi"
}

df = df.rename(columns=rename_dict)
df["Layer"] = pd.to_numeric(df["Layer"], errors="coerce")

depth_map = {
    1: "0–20 cm",
    2: "20–50 cm",
    3: "50–80 cm"
}
df["Depth"] = df["Layer"].map(depth_map)

# ------------------------------------------------------------
# 4. VARIABLE SELECTION
# ------------------------------------------------------------
active_vars = [
    "EC12",
    "ESP",
    "pH",
    "SOC",
    "Clay",
    "CEC",
    "TN"
]

supplementary_vars = [
    "TotalBacteria",
    "TotalFungi",
    "TypesBacteria",
    "TypesFungi"
]

for col in active_vars + supplementary_vars:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

pca_df = df[["Zone", "Layer", "Depth"] + active_vars + supplementary_vars].dropna(
    subset=active_vars
).copy()

print("PCA dataset shape:", pca_df.shape)

print("\nN by Zone:")
print(pca_df["Zone"].value_counts())

print("\nN by Depth:")
print(pca_df["Depth"].value_counts())

print("\nN by Zone x Depth:")
print(pd.crosstab(pca_df["Zone"], pca_df["Depth"]))

# ------------------------------------------------------------
# 5. DISPLAY LABELS
# ------------------------------------------------------------
active_labels = {
    "EC12": r"EC$_{1:2}$",
    "ESP": "ESP",
    "pH": "pH",
    "SOC": "SOC",
    "Clay": "Clay",
    "CEC": "CEC",
    "TN": "TN"
}

supplementary_labels = {
    "TotalBacteria": "Ba",
    "TotalFungi": "Fa",
    "TypesBacteria": "Br",
    "TypesFungi": "Fr"
}

zone_palette = {
    "PERKERRA": "#4C72B0",
    "TANA": "#DD8452"
}

depth_palette = {
    "0–20 cm": "#1B9E77",
    "20–50 cm": "#D95F02",
    "50–80 cm": "#7570B3"
}

# ------------------------------------------------------------
# 6. FONT AND SIZE SETTINGS
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 17,
    "axes.labelsize": 19,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 18,
    "legend.title_fontsize": 19,
    "axes.linewidth": 1.0
})

FONT_AXIS = 19
FONT_TICK = 16
FONT_PANEL = 23
FONT_LEGEND = 18
FONT_LEGEND_TITLE = 19
FONT_ARROW_GLOBAL = 11.5
FONT_ARROW_ZONE = 11

# Bigger points
MARKER_GLOBAL = 62
MARKER_ZONE = 56
LEGEND_MARKER = 13

# ------------------------------------------------------------
# 7. PCA FUNCTIONS
# ------------------------------------------------------------
def compute_pca(data, variables):
    X = data[variables].copy()

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA()
    scores = pca.fit_transform(X_scaled)

    scores_df = pd.DataFrame(
        scores,
        columns=[f"PC{i+1}" for i in range(scores.shape[1])]
    )

    meta_cols = [c for c in ["Zone", "Layer", "Depth"] if c in data.columns]

    scores_df = pd.concat(
        [data[meta_cols].reset_index(drop=True), scores_df],
        axis=1
    )

    loadings = pd.DataFrame(
        pca.components_.T,
        index=variables,
        columns=[f"PC{i+1}" for i in range(len(variables))]
    )

    explained_df = pd.DataFrame({
        "PC": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
        "Explained_variance_ratio": pca.explained_variance_ratio_,
        "Explained_variance_pct": pca.explained_variance_ratio_ * 100,
        "Cumulative_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
        "Cumulative_variance_pct": np.cumsum(pca.explained_variance_ratio_) * 100
    })

    return pca, scaler, scores_df, loadings, explained_df


def compute_supplementary_correlations(data, scores_df, supplementary_variables):
    rows = []

    data_reset = data.reset_index(drop=True).copy()
    scores_reset = scores_df.reset_index(drop=True).copy()

    for var in supplementary_variables:
        valid = data_reset[var].notna().to_numpy()

        if valid.sum() < 3:
            continue

        y = data_reset.loc[valid, var].to_numpy()
        pc1 = scores_reset.loc[valid, "PC1"].to_numpy()
        pc2 = scores_reset.loc[valid, "PC2"].to_numpy()

        r_pc1, p_pc1 = pearsonr(pc1, y)
        r_pc2, p_pc2 = pearsonr(pc2, y)

        rows.append({
            "Variable": var,
            "PC1_r": r_pc1,
            "PC1_p": p_pc1,
            "PC2_r": r_pc2,
            "PC2_p": p_pc2
        })

    return pd.DataFrame(rows)


def add_confidence_ellipse(
    x, y, ax, n_std=1.5,
    facecolor="none", edgecolor="black",
    alpha=0.12, linewidth=1.0
):
    if len(x) < 3:
        return

    cov = np.cov(x, y)

    if np.any(np.isnan(cov)) or np.linalg.det(cov) <= 0:
        return

    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]

    vals = vals[order]
    vecs = vecs[:, order]

    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)

    ellipse = Ellipse(
        xy=(np.mean(x), np.mean(y)),
        width=width,
        height=height,
        angle=theta,
        facecolor=facecolor,
        edgecolor=edgecolor,
        alpha=alpha,
        linewidth=linewidth
    )

    ax.add_patch(ellipse)


def add_loading_arrows(
    ax,
    loadings,
    labels,
    scale=3.0,
    color="black",
    fontsize=10,
    linewidth=1.25,
    offsets=None,
    fixed_label_positions=None,
    zorder=4
):
    if offsets is None:
        offsets = {}

    if fixed_label_positions is None:
        fixed_label_positions = {}

    for var in loadings.index:
        x = loadings.loc[var, "PC1"] * scale
        y = loadings.loc[var, "PC2"] * scale

        ax.annotate(
            "",
            xy=(x, y),
            xytext=(0, 0),
            arrowprops=dict(
                arrowstyle="-|>",
                color=color,
                lw=linewidth,
                shrinkA=0,
                shrinkB=0,
                mutation_scale=13
            ),
            zorder=zorder
        )

        if var in fixed_label_positions:
            text_x, text_y = fixed_label_positions[var]
        else:
            dx, dy = offsets.get(var, (0.08, 0.08))
            text_x, text_y = x + dx, y + dy

        ax.text(
            text_x,
            text_y,
            labels.get(var, var),
            fontsize=fontsize,
            color=color,
            ha="center",
            va="center",
            zorder=zorder + 1,
            bbox=dict(
                boxstyle="round,pad=0.16",
                facecolor="white",
                edgecolor="none",
                alpha=0.92
            )
        )


def add_supplementary_arrows(
    ax,
    supp_corr,
    labels,
    scale=2.6,
    color="darkred",
    fontsize=10,
    linewidth=1.20,
    offsets=None,
    fixed_label_positions=None,
    zorder=5
):
    if supp_corr is None or supp_corr.empty:
        return

    if offsets is None:
        offsets = {}

    if fixed_label_positions is None:
        fixed_label_positions = {}

    for _, row in supp_corr.iterrows():
        var = row["Variable"]

        x = row["PC1_r"] * scale
        y = row["PC2_r"] * scale

        ax.annotate(
            "",
            xy=(x, y),
            xytext=(0, 0),
            arrowprops=dict(
                arrowstyle="-|>",
                color=color,
                lw=linewidth,
                shrinkA=0,
                shrinkB=0,
                mutation_scale=13
            ),
            zorder=zorder
        )

        if var in fixed_label_positions:
            text_x, text_y = fixed_label_positions[var]
        else:
            dx, dy = offsets.get(var, (0.08, 0.08))
            text_x, text_y = x + dx, y + dy

        ax.text(
            text_x,
            text_y,
            labels.get(var, var),
            fontsize=fontsize,
            color=color,
            ha="center",
            va="center",
            zorder=zorder + 1,
            bbox=dict(
                boxstyle="round,pad=0.15",
                facecolor="white",
                edgecolor="none",
                alpha=0.92
            )
        )


def set_axis_limits_from_data(
    ax,
    scores_df,
    loadings=None,
    loadings_scale=1.0,
    supp_corr=None,
    supp_scale=1.0,
    fixed_label_positions=None,
    pad=0.25
):
    xs = list(scores_df["PC1"].dropna().values)
    ys = list(scores_df["PC2"].dropna().values)

    if loadings is not None:
        xs.extend((loadings["PC1"] * loadings_scale).values)
        ys.extend((loadings["PC2"] * loadings_scale).values)

    if supp_corr is not None and not supp_corr.empty:
        xs.extend((supp_corr["PC1_r"] * supp_scale).values)
        ys.extend((supp_corr["PC2_r"] * supp_scale).values)

    if fixed_label_positions is not None:
        for x, y in fixed_label_positions.values():
            xs.append(x)
            ys.append(y)

    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    dx = (x_max - x_min) * pad
    dy = (y_max - y_min) * pad

    ax.set_xlim(x_min - dx, x_max + dx)
    ax.set_ylim(y_min - dy, y_max + dy)


def format_axis(ax, pc1_pct, pc2_pct):
    ax.axhline(0, color="0.55", linewidth=0.9, zorder=0)
    ax.axvline(0, color="0.55", linewidth=0.9, zorder=0)

    ax.set_xlabel(f"PC1 ({pc1_pct:.1f}%)", fontsize=FONT_AXIS)
    ax.set_ylabel(f"PC2 ({pc2_pct:.1f}%)", fontsize=FONT_AXIS)

    ax.tick_params(axis="both", labelsize=FONT_TICK)

    for spine in ax.spines.values():
        spine.set_linewidth(1.0)

# ------------------------------------------------------------
# 8. COMPUTE GLOBAL PCA
# ------------------------------------------------------------
pca_global, scaler_global, scores_global, loadings_global, explained_global = compute_pca(
    pca_df,
    active_vars
)

supp_global = compute_supplementary_correlations(
    pca_df,
    scores_global,
    supplementary_vars
)

# ------------------------------------------------------------
# 9. COMPUTE PCA BY ZONE
# ------------------------------------------------------------
pca_results_by_zone = {}

for zone in ["PERKERRA", "TANA"]:
    sub = pca_df[pca_df["Zone"] == zone].copy()

    pca_z, scaler_z, scores_z, loadings_z, explained_z = compute_pca(
        sub,
        active_vars
    )

    pca_results_by_zone[zone] = {
        "pca": pca_z,
        "scaler": scaler_z,
        "scores": scores_z,
        "loadings": loadings_z,
        "explained": explained_z
    }

# ------------------------------------------------------------
# 10. SAVE TABLES
# ------------------------------------------------------------
explained_global.to_excel(
    os.path.join(tables_dir, "PCA_global_explained_variance.xlsx"),
    index=False
)

loadings_global.to_excel(
    os.path.join(tables_dir, "PCA_global_loadings.xlsx")
)

scores_global.to_excel(
    os.path.join(tables_dir, "PCA_global_scores.xlsx"),
    index=False
)

supp_global.to_excel(
    os.path.join(tables_dir, "PCA_global_supplementary_microbiology_correlations.xlsx"),
    index=False
)

for zone, res in pca_results_by_zone.items():
    res["explained"].to_excel(
        os.path.join(tables_dir, f"PCA_{zone}_explained_variance.xlsx"),
        index=False
    )

    res["loadings"].to_excel(
        os.path.join(tables_dir, f"PCA_{zone}_loadings.xlsx")
    )

    res["scores"].to_excel(
        os.path.join(tables_dir, f"PCA_{zone}_scores.xlsx"),
        index=False
    )

summary_rows = []

summary_rows.append({
    "PCA": "Global",
    "PC1_explained_pct": explained_global.loc[0, "Explained_variance_pct"],
    "PC2_explained_pct": explained_global.loc[1, "Explained_variance_pct"],
    "PC1_PC2_cumulative_pct": explained_global.loc[:1, "Explained_variance_pct"].sum()
})

for zone, res in pca_results_by_zone.items():
    summary_rows.append({
        "PCA": zone,
        "PC1_explained_pct": res["explained"].loc[0, "Explained_variance_pct"],
        "PC2_explained_pct": res["explained"].loc[1, "Explained_variance_pct"],
        "PC1_PC2_cumulative_pct": res["explained"].loc[:1, "Explained_variance_pct"].sum()
    })

pca_summary = pd.DataFrame(summary_rows)

pca_summary.to_excel(
    os.path.join(tables_dir, "PCA_summary_global_and_by_zone.xlsx"),
    index=False
)

print("\nPCA summary:")
print(pca_summary)

# ------------------------------------------------------------
# 11. PLOT COMBINED FIGURE
# ------------------------------------------------------------
fig = plt.figure(figsize=(20.2, 14.8))

gs = fig.add_gridspec(
    nrows=2,
    ncols=6,
    height_ratios=[1.58, 1.08],
    width_ratios=[1, 1, 1, 1, 1, 1],
    hspace=0.36,
    wspace=0.55
)

ax_a = fig.add_subplot(gs[0, 1:5])
ax_b = fig.add_subplot(gs[1, 0:3])
ax_c = fig.add_subplot(gs[1, 3:6])

# ------------------------------------------------------------
# Panel a — Global PCA
# ------------------------------------------------------------
for zone, color in zone_palette.items():
    sub = scores_global[scores_global["Zone"] == zone]

    ax_a.scatter(
        sub["PC1"],
        sub["PC2"],
        s=MARKER_GLOBAL,
        color=color,
        alpha=0.86,
        edgecolor="none",
        label=zone,
        zorder=2
    )

    add_confidence_ellipse(
        sub["PC1"].values,
        sub["PC2"].values,
        ax=ax_a,
        n_std=1.5,
        facecolor=color,
        edgecolor=color,
        alpha=0.13,
        linewidth=1.15
    )

global_loading_offsets = {
    "EC12": (0.28, 0.02),
    "ESP": (-0.22, 0.22),
    "pH": (0.25, -0.12),
    "SOC": (-0.24, 0.20),
    "Clay": (0.25, 0.18),
    "CEC": (0.25, 0.25),
    "TN": (-0.18, 0.27)
}

global_supp_offsets = {
    "TotalBacteria": (-0.30, 0.20),
    "TypesBacteria": (-0.22, 0.02),
    "TotalFungi": (0.22, 0.02),
    "TypesFungi": (0.30, 0.13)
}

add_loading_arrows(
    ax=ax_a,
    loadings=loadings_global,
    labels=active_labels,
    scale=3.35,
    color="black",
    fontsize=FONT_ARROW_GLOBAL,
    linewidth=1.25,
    offsets=global_loading_offsets
)

add_supplementary_arrows(
    ax=ax_a,
    supp_corr=supp_global,
    labels=supplementary_labels,
    scale=2.70,
    color="darkred",
    fontsize=FONT_ARROW_GLOBAL,
    linewidth=1.20,
    offsets=global_supp_offsets
)

set_axis_limits_from_data(
    ax=ax_a,
    scores_df=scores_global,
    loadings=loadings_global,
    loadings_scale=3.35,
    supp_corr=supp_global,
    supp_scale=2.70,
    pad=0.26
)

format_axis(
    ax_a,
    explained_global.loc[0, "Explained_variance_pct"],
    explained_global.loc[1, "Explained_variance_pct"]
)

ax_a.text(
    -0.09,
    1.045,
    "a)",
    transform=ax_a.transAxes,
    fontsize=FONT_PANEL,
    fontweight="bold",
    ha="left",
    va="bottom"
)

# ------------------------------------------------------------
# Legends — bigger and slightly more separated
# ------------------------------------------------------------
zone_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor=zone_palette[z],
        markersize=LEGEND_MARKER,
        label=z
    )
    for z in ["PERKERRA", "TANA"]
]

depth_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor=depth_palette[d],
        markersize=LEGEND_MARKER,
        label=d
    )
    for d in ["0–20 cm", "20–50 cm", "50–80 cm"]
]

legend_zone = ax_a.legend(
    handles=zone_handles,
    title="Zone",
    loc="upper left",
    bbox_to_anchor=(1.06, 0.98),
    frameon=False,
    borderaxespad=0.0,
    handletextpad=0.6,
    labelspacing=0.8,
    fontsize=FONT_LEGEND,
    title_fontsize=FONT_LEGEND_TITLE
)

ax_a.add_artist(legend_zone)

ax_a.legend(
    handles=depth_handles,
    title="Depth",
    loc="upper left",
    bbox_to_anchor=(1.06, 0.51),
    frameon=False,
    borderaxespad=0.0,
    handletextpad=0.6,
    labelspacing=0.8,
    fontsize=FONT_LEGEND,
    title_fontsize=FONT_LEGEND_TITLE
)

# ------------------------------------------------------------
# Panel b — Perkerra PCA
# ------------------------------------------------------------
scores_perkerra = pca_results_by_zone["PERKERRA"]["scores"]
loadings_perkerra = pca_results_by_zone["PERKERRA"]["loadings"]
explained_perkerra = pca_results_by_zone["PERKERRA"]["explained"]

for depth, color in depth_palette.items():
    sub = scores_perkerra[scores_perkerra["Depth"] == depth]

    ax_b.scatter(
        sub["PC1"],
        sub["PC2"],
        s=MARKER_ZONE,
        color=color,
        alpha=0.86,
        edgecolor="none",
        zorder=2
    )

perkerra_offsets = {
    "EC12": (0.23, 0.06),
    "ESP": (0.22, 0.22),
    "pH": (0.28, -0.15),
    "SOC": (-0.28, 0.28),
    "Clay": (0.08, -0.22),
    "CEC": (0.25, 0.25),
    "TN": (-0.34, -0.18)
}

# TN is forced to appear lower in panel b
perkerra_fixed_label_positions = {
    "TN": (-1.45, 0.65)
}

add_loading_arrows(
    ax=ax_b,
    loadings=loadings_perkerra,
    labels=active_labels,
    scale=2.65,
    color="black",
    fontsize=FONT_ARROW_ZONE,
    linewidth=1.15,
    offsets=perkerra_offsets,
    fixed_label_positions=perkerra_fixed_label_positions
)

set_axis_limits_from_data(
    ax=ax_b,
    scores_df=scores_perkerra,
    loadings=loadings_perkerra,
    loadings_scale=2.65,
    fixed_label_positions=perkerra_fixed_label_positions,
    pad=0.28
)

format_axis(
    ax_b,
    explained_perkerra.loc[0, "Explained_variance_pct"],
    explained_perkerra.loc[1, "Explained_variance_pct"]
)

ax_b.text(
    -0.09,
    1.045,
    "b)",
    transform=ax_b.transAxes,
    fontsize=FONT_PANEL,
    fontweight="bold",
    ha="left",
    va="bottom"
)

# ------------------------------------------------------------
# Panel c — Tana PCA
# ------------------------------------------------------------
scores_tana = pca_results_by_zone["TANA"]["scores"]
loadings_tana = pca_results_by_zone["TANA"]["loadings"]
explained_tana = pca_results_by_zone["TANA"]["explained"]

for depth, color in depth_palette.items():
    sub = scores_tana[scores_tana["Depth"] == depth]

    ax_c.scatter(
        sub["PC1"],
        sub["PC2"],
        s=MARKER_ZONE,
        color=color,
        alpha=0.86,
        edgecolor="none",
        zorder=2
    )

tana_offsets = {
    "EC12": (0.21, 0.05),
    "ESP": (-0.17, -0.22),
    "pH": (0.18, -0.21),
    "SOC": (-0.22, 0.23),
    "Clay": (0.20, 0.20),
    "CEC": (0.22, -0.18),
    "TN": (0.20, 0.25)
}

add_loading_arrows(
    ax=ax_c,
    loadings=loadings_tana,
    labels=active_labels,
    scale=2.65,
    color="black",
    fontsize=FONT_ARROW_ZONE,
    linewidth=1.15,
    offsets=tana_offsets
)

set_axis_limits_from_data(
    ax=ax_c,
    scores_df=scores_tana,
    loadings=loadings_tana,
    loadings_scale=2.65,
    pad=0.28
)

format_axis(
    ax_c,
    explained_tana.loc[0, "Explained_variance_pct"],
    explained_tana.loc[1, "Explained_variance_pct"]
)

ax_c.text(
    -0.09,
    1.045,
    "c)",
    transform=ax_c.transAxes,
    fontsize=FONT_PANEL,
    fontweight="bold",
    ha="left",
    va="bottom"
)

# ------------------------------------------------------------
# 12. SAVE FIGURE
# ------------------------------------------------------------
fig_path_png = os.path.join(figures_dir, "figure_4_13_pca_soil_gradients.png")
fig_path_tiff = os.path.join(figures_dir, "figure_4_13_pca_soil_gradients.tiff")
fig_path_pdf = os.path.join(figures_dir, "figure_4_13_pca_soil_gradients.pdf")

plt.subplots_adjust(
    left=0.07,
    right=0.78,
    top=0.96,
    bottom=0.08
)

plt.savefig(fig_path_png, dpi=600, bbox_inches="tight")
plt.savefig(fig_path_tiff, dpi=600, bbox_inches="tight")
plt.savefig(fig_path_pdf, bbox_inches="tight")

plt.show()

print("\nCombined PCA figure saved as:")
print(fig_path_png)
print(fig_path_tiff)
print(fig_path_pdf)

print("\nAll PCA tables saved in:")
print(tables_dir)
